# 07 — Domain-Adaptive Pretraining (DAP)

**Arm 2, stage A.** Continues **masked-language-model (MLM) pretraining** on each target
domain's *unlabeled* train text, starting from the CoNLL-2003 baseline encoder trained in
Notebook 04. No labels are used — that is the whole point of DAP (Gururangan et al., 2020).
Notebook 08 then few-shot-fine-tunes these adapted encoders on the same splits Notebook 05
uses, so **NB05 (Arm 1) vs NB08 (Arm 2)** isolates exactly one variable: *did the encoder see
unlabeled target text before few-shot fine-tuning?*

This notebook is wired to the **executed pipeline** (Notebooks 03–06):
- reads the baseline from `models/baseline_conll2003/`
- reads raw text from `<ds>/<ds>_train.jsonl` (tokens only; tags ignored)
- writes adapted encoders to `models/dap_wnut17/` and `models/dap_scierc/`
- writes `results/dap_training_info.json` (perplexity + wall-clock, for NB10's cost story)

One HF behavior we rely on: the baseline checkpoint was saved by a **token-classification**
model, but we reload it with `AutoModelForMaskedLM`. HF keeps the shared BERT encoder (which
carries the CoNLL fine-tuning) and **re-initializes the MLM head** while dropping the unused
NER classifier — expected, not a bug. We print the missing/unexpected keys so it is visible.

In [1]:
!pip -q install "transformers==4.44.2" "datasets==2.19.2" "accelerate>=0.26.0"
import torch
print('GPU available:', torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 62.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
GPU available: True


## Step 1 — Mount Drive and configure

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json
from pathlib import Path

# --- executed-pipeline layout (identical to Notebooks 03-06) ---
PROCESSED      = Path('/content/drive/MyDrive/AAI590/data/processed')
TOKENIZED_DIR  = PROCESSED / 'tokenized'         # HF datasets saved by NB03
LABELS_DIR     = PROCESSED / 'label_maps'        # <ds>_label_map.json (NB03)
MODELS_DIR     = PROCESSED / 'models'            # baseline_conll2003 (NB04)
RESULTS_DIR    = PROCESSED / 'results'
FEWSHOT_SPLITS = PROCESSED / 'fewshot_splits'    # raw jsonl demos (NB02)
BASELINE_DIR   = MODELS_DIR / 'baseline_conll2003'

# shared experiment grid (identical to NB05/NB06)
TARGET_DATASETS = ['wnut17', 'scierc']
BUDGETS = [50, 100, 200]
SEEDS   = [13, 42, 101]

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def load_label_map(dataset_name):
    with open(LABELS_DIR / f'{dataset_name}_label_map.json') as f:
        m = json.load(f)
    label2id = {str(k): int(v) for k, v in m['label2id'].items()}
    id2label = {int(k): str(v) for k, v in m['id2label'].items()}
    return label2id, id2label

print('processed dir :', PROCESSED)
print('baseline dir  :', BASELINE_DIR, '(exists:', BASELINE_DIR.exists(), ')')

Mounted at /content/drive
processed dir : /content/drive/MyDrive/AAI590/data/processed
baseline dir  : /content/drive/MyDrive/AAI590/data/processed/models/baseline_conll2003 (exists: True )


In [3]:
from transformers import set_seed

# DAP hyperparameters. Corpora are tiny (~3.4k WNUT / ~1.9k SciERC sentences), so a few
# epochs is enough to nudge the encoder toward target vocabulary/style without overfitting.
DAP_EPOCHS     = 5
LEARNING_RATE  = 5e-5
BATCH_SIZE     = 32
MLM_PROBABILITY = 0.15
MAX_LENGTH     = 256
DAP_SEED       = 42

assert BASELINE_DIR.exists(), 'Run Notebook 04 first (models/baseline_conll2003 missing).'

## Step 2 — Train one DAP checkpoint per target domain

In [4]:
import math, time
import pandas as pd
from datasets import Dataset
from transformers import (AutoModelForMaskedLM, AutoTokenizer,
                          DataCollatorForLanguageModeling, Trainer, TrainingArguments)

def make_mlm_dataset(rows, tokenizer):
    # token sequences only -- tags deliberately ignored (DAP is label-free)
    def encode(batch):
        return tokenizer(batch['tokens'], is_split_into_words=True,
                         truncation=True, max_length=MAX_LENGTH)
    ds = Dataset.from_list([{'tokens': r['tokens']} for r in rows])
    return ds.map(encode, batched=True, remove_columns=['tokens'])

dap_info = {}
for ds_name in TARGET_DATASETS:
    print('\n' + '=' * 70)
    print('DAP on', ds_name)
    print('=' * 70)

    train_rows = load_jsonl(PROCESSED / ds_name / f'{ds_name}_train.jsonl')
    val_rows   = load_jsonl(PROCESSED / ds_name / f'{ds_name}_validation.jsonl')

    tokenizer = AutoTokenizer.from_pretrained(str(BASELINE_DIR))
    train_ds = make_mlm_dataset(train_rows, tokenizer)
    val_ds   = make_mlm_dataset(val_rows, tokenizer)

    set_seed(DAP_SEED)
    model, info = AutoModelForMaskedLM.from_pretrained(str(BASELINE_DIR),
                                                       output_loading_info=True)
    print('freshly initialized (expected -- new MLM head):', info['missing_keys'])
    print('dropped from checkpoint (expected -- old NER head):', info['unexpected_keys'])

    args = TrainingArguments(
        output_dir=f'/content/dap_{ds_name}_checkpoints',
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=DAP_EPOCHS,
        weight_decay=0.01,
        eval_strategy='epoch',
        logging_strategy='epoch',
        save_strategy='no',
        report_to='none',
        seed=DAP_SEED,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer,
                                                      mlm_probability=MLM_PROBABILITY),
        tokenizer=tokenizer,
    )

    t0 = time.time()
    trainer.train()
    train_seconds = time.time() - t0

    # eval perplexity per epoch -- so the epoch count is a checked decision, not a guess
    ppl_rows = [{'epoch': h['epoch'], 'eval_loss': h['eval_loss'],
                 'perplexity': math.exp(h['eval_loss'])}
                for h in trainer.state.log_history if 'eval_loss' in h]
    print(pd.DataFrame(ppl_rows).round(3).to_string(index=False))

    out_dir = MODELS_DIR / f'dap_{ds_name}'
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(out_dir))
    tokenizer.save_pretrained(str(out_dir))
    print(f'saved -> {out_dir}  ({train_seconds:.0f}s)')

    dap_info[ds_name] = {
        'checkpoint_dir': str(out_dir),
        'epochs': DAP_EPOCHS,
        'train_sentences': len(train_rows),
        'train_seconds': train_seconds,
        'eval_perplexity_by_epoch': ppl_rows,
    }

    del trainer, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


DAP on wnut17


Map:   0%|          | 0/3394 [00:00<?, ? examples/s]

Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

Some weights of BertForMaskedLM were not initialized from the model checkpoint at /content/drive/MyDrive/AAI590/data/processed/models/baseline_conll2003 and are newly initialized: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


freshly initialized (expected -- new MLM head): ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
dropped from checkpoint (expected -- old NER head): ['classifier.bias', 'classifier.weight']


...

Map:   0%|          | 0/275 [00:00<?, ? examples/s]

Some weights of BertForMaskedLM were not initialized from the model checkpoint at /content/drive/MyDrive/AAI590/data/processed/models/baseline_conll2003 and are newly initialized: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


freshly initialized (expected -- new MLM head): ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
dropped from checkpoint (expected -- old NER head): ['classifier.bias', 'classifier.weight']


Epoch,Training Loss,Validation Loss
1,6.439000,5.125872
2,4.733600,4.252526
3,4.201700,4.050994
4,3.917700,3.897631
5,3.874900,3.904166


 epoch  eval_loss  perplexity
   1.0      5.126     168.321
   2.0      4.253      70.283
   3.0      4.051      57.455
   4.0      3.898      49.286
   5.0      3.904      49.609
saved -> /content/drive/MyDrive/AAI590/data/processed/models/dap_scierc  (164s)


## Step 3 — Save DAP training info

The per-domain wall-clock is amortized into Arm 2's cost in Notebook 10 (DAP is compute Arm 1
never pays), so we persist it alongside the other results.

In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_fp = RESULTS_DIR / 'dap_training_info.json'
with open(out_fp, 'w') as f:
    json.dump(dap_info, f, indent=2)
print('saved ->', out_fp)

saved -> /content/drive/MyDrive/AAI590/data/processed/results/dap_training_info.json


**How to read the perplexity tables:** if eval perplexity is still clearly falling at the
last epoch there is headroom for a couple more; if it bottoms out and turns back up the model
has started overfitting the tiny corpus and `DAP_EPOCHS` should come down.